[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/06_linear_regression/08_diagnostics_and_failure_cases.ipynb)

# Bài 08 — Chẩn đoán model và các trường hợp Linear Regression thất bại

**🎯 Mục tiêu**
- Đọc **residual plot**, Q-Q plot để kiểm tra các giả định.
- Nhận diện và xử lý 6 failure case: phi tuyến, phương sai không đều, outlier & điểm đòn bẩy, đa cộng tuyến, ngoại suy, thiếu biến quan trọng.
- Checklist chẩn đoán dùng cho mọi model hồi quy.

> Một model có R² cao **chưa chắc** đúng. Luôn nhìn vào **residual** — phần model chưa giải thích được.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/06_linear_regression

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import HuberRegressor, LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

from ml_course.visualization import plot_residuals, set_style

set_style()
rng = np.random.default_rng(0)

## 1. Model "đạt chuẩn": residual trông như nhiễu ngẫu nhiên

Khi các giả định được thoả: residual vs fitted là **dải ngang đều quanh 0**, histogram hình chuông, Q-Q plot gần đường thẳng.

In [ ]:
x = rng.uniform(0, 10, 300)
y = 2 + 1.5 * x + rng.normal(0, 1, 300)
good = LinearRegression().fit(x[:, None], y)
plot_residuals(y, good.predict(x[:, None]), title="Trường hợp tốt — residual không có cấu trúc")
plt.show()

## 2. Failure case 1 — Quan hệ phi tuyến

In [ ]:
y_nl = 2 + 0.3 * x**2 + rng.normal(0, 1, 300)
lin = LinearRegression().fit(x[:, None], y_nl)
print(f"R² = {r2_score(y_nl, lin.predict(x[:, None])):.3f} — trông 'khá cao' nhưng...")
plot_residuals(y_nl, lin.predict(x[:, None]), title="Phi tuyến: residual có hình chữ U")
plt.show()

**Dấu hiệu:** residual vs fitted có **hình cong** (chữ U). **Khắc phục:** thêm feature phi tuyến ($x^2$, $\log x$, spline — bài 05, chương 17), hoặc dùng model phi tuyến.

## 3. Failure case 2 — Phương sai không đều (heteroscedasticity)

Rất phổ biến với dữ liệu tiền bạc: nhà càng đắt, sai số tuyệt đối càng lớn.

In [ ]:
df = pd.read_csv("../../data/sample/house_prices.csv")
features = ["area_m2", "bedrooms", "bathrooms", "age_years", "distance_to_center_km"]
X = pd.get_dummies(df[features + ["district"]], columns=["district"], drop_first=True, dtype=float)
price = df["price_billion_vnd"]

m_raw = LinearRegression().fit(X, price)
plot_residuals(price, m_raw.predict(X), title="Target gốc: residual 'loe' hình phễu + lệch")
plt.show()

**Dấu hiệu:** residual "loe" ra khi fitted tăng (hình phễu). Hệ quả: model bị chi phối bởi các căn nhà đắt; khoảng tin cậy sai.

**Khắc phục:** biến đổi target — **log(giá)**. Trên thang log, sai số trở thành **tương đối** (%), và quan hệ nhân (giá = diện tích × hệ số quận × ...) trở thành cộng:

In [ ]:
X_log = X.copy()
X_log["area_m2"] = np.log(X_log["area_m2"])
m_log = LinearRegression().fit(X_log, np.log(price))
plot_residuals(np.log(price), m_log.predict(X_log), title="log(giá) ~ log(diện tích) + ...: residual đều hơn nhiều")
plt.show()

pred_back = np.exp(m_log.predict(X_log))
print(f"MAE (target gốc)    : {mean_absolute_error(price, m_raw.predict(X)):.3f} tỷ")
print(f"MAE (log rồi exp lại): {mean_absolute_error(price, pred_back):.3f} tỷ")

> 💡 Khi dự đoán trên thang log rồi `exp` ngược lại, ta nhận được **median** (không phải mean) của phân phối dự đoán — thường là điều ta muốn với dữ liệu giá. Đây là kỹ thuật dùng rất nhiều cho giá cả, doanh thu, số lượng.

## 4. Failure case 3 — Outlier và điểm đòn bẩy (leverage)

MSE **bình phương** sai số → một điểm sai rất xa có thể kéo lệch cả đường hồi quy. Nguy hiểm nhất là điểm vừa có **y bất thường** vừa có **x ở xa** (đòn bẩy cao).

In [ ]:
x_o = np.r_[rng.uniform(0, 10, 50), 25.0]
y_o = np.r_[1 + 0.8 * x_o[:50] + rng.normal(0, 0.8, 50), 2.0]       # 1 điểm ở xa với y bất thường
grid = np.linspace(0, 25, 100)
ols = LinearRegression().fit(x_o[:, None], y_o)
huber = HuberRegressor().fit(x_o[:, None], y_o)
ols_clean = LinearRegression().fit(x_o[:50, None], y_o[:50])

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.scatter(x_o[:50], y_o[:50], s=15)
ax.scatter(x_o[50:], y_o[50:], s=80, color="red", label="1 điểm đòn bẩy cao")
ax.plot(grid, ols.predict(grid[:, None]), label=f"OLS (slope = {ols.coef_[0]:.2f})")
ax.plot(grid, huber.predict(grid[:, None]), label=f"Huber (slope = {huber.coef_[0]:.2f})")
ax.plot(grid, ols_clean.predict(grid[:, None]), "k--", label=f"OLS không có điểm đó (slope = {ols_clean.coef_[0]:.2f})")
ax.legend()
ax.set_title("Một điểm duy nhất bẻ cong đường OLS")
plt.show()

**Phát hiện:** residual lớn, leverage cao (hat matrix $H = X(X^\top X)^{-1}X^\top$, phần tử đường chéo $h_{ii}$), **Cook's distance** (đo ảnh hưởng của việc bỏ từng điểm). **Khắc phục:** kiểm tra có phải lỗi dữ liệu (chương 03); dùng loss bền vững — **Huber** (bình phương khi sai số nhỏ, tuyến tính khi lớn) hoặc MAE; log-transform.

In [ ]:
Xd = np.c_[np.ones(len(x_o)), x_o]
H = Xd @ np.linalg.inv(Xd.T @ Xd) @ Xd.T
leverage = np.diag(H)
resid = y_o - Xd @ np.linalg.lstsq(Xd, y_o, rcond=None)[0]
mse_ = resid @ resid / (len(y_o) - 2)
cooks = resid**2 / (2 * mse_) * leverage / (1 - leverage) ** 2
print("Leverage cao nhất:", leverage.max().round(3), "tại điểm", leverage.argmax())
print("Cook's distance cao nhất:", cooks.max().round(2), "(ngưỡng kinh nghiệm: > 4/n =", round(4 / len(y_o), 3), ")")

## 5. Failure case 4 — Đa cộng tuyến (multicollinearity)

Khi các feature tương quan rất mạnh với nhau, hệ số trở nên **không ổn định**: thay đổi nhỏ trong dữ liệu làm hệ số thay đổi lớn (thậm chí đổi dấu), dù **dự đoán** vẫn tốt.

**VIF (Variance Inflation Factor)** của feature $j$: $\text{VIF}_j = \frac{1}{1 - R^2_j}$, với $R^2_j$ là R² khi hồi quy $x_j$ theo các feature còn lại. VIF > 5–10 là dấu hiệu đa cộng tuyến.

In [ ]:
def vif(df_features):
    out = {}
    for col in df_features.columns:
        others = df_features.drop(columns=col)
        r2 = LinearRegression().fit(others, df_features[col]).score(others, df_features[col])
        out[col] = 1 / (1 - r2) if r2 < 1 else np.inf
    return pd.Series(out).round(2)


feat = df[features].copy()
feat["area_feet2"] = feat["area_m2"] * 10.764 + rng.normal(0, 5, len(feat))   # gần như trùng diện tích
print(vif(feat))

coefs = []
for seed in range(5):
    idx = np.random.default_rng(seed).choice(len(feat), 400, replace=False)
    coefs.append(LinearRegression().fit(feat.iloc[idx], price.iloc[idx]).coef_[[0, -1]])
print("\nHệ số (area_m2, area_feet2) trên 5 mẫu con khác nhau:\n", np.round(coefs, 3))

Hai hệ số của diện tích "giằng co" nhau và thay đổi mạnh giữa các mẫu con — nhưng tổng ảnh hưởng của diện tích vẫn ổn. **Khắc phục:** bỏ feature dư thừa, gộp feature, PCA, hoặc **Ridge** (bài 07). Nếu chỉ cần dự đoán (không cần diễn giải từng hệ số), đa cộng tuyến ít nguy hại hơn.

## 6. Failure case 5 — Ngoại suy

Model tuyến tính kéo dài đường thẳng **vô hạn**. Ví dụ quan hệ **lợi suất giảm dần** rất phổ biến: chi phí quảng cáo càng cao, doanh số tăng thêm càng ít, rồi bão hoà. Nếu dữ liệu train chỉ có vùng chi tiêu thấp (gần như tuyến tính), model sẽ ngoại suy sai nghiêm trọng:

In [ ]:
spend = rng.uniform(0, 40, 80)                                         # dữ liệu train: chi tiêu 0–40 (triệu)
true_sales = lambda s: 100 * (1 - np.exp(-s / 60))                     # quy luật thật: bão hoà ở 100
sales = true_sales(spend) + rng.normal(0, 2, 80)
m_ads = LinearRegression().fit(spend[:, None], sales)
for s in [20, 40, 150, 300]:
    pred = m_ads.predict([[s]])[0]
    flag = "  ← NGOẠI SUY" if s > 40 else ""
    print(f"chi {s:3d}: dự đoán {pred:6.1f} | thật {true_sales(s):6.1f}{flag}")

grid_s = np.linspace(0, 300, 200)
fig, ax = plt.subplots(figsize=(8, 4))
ax.scatter(spend, sales, s=12, label="train (chi 0–40)")
ax.plot(grid_s, true_sales(grid_s), "k--", label="quy luật thật (bão hoà)")
ax.plot(grid_s, m_ads.predict(grid_s[:, None]), "r", label="Linear Regression")
ax.axvspan(40, 300, color="gray", alpha=0.15)
ax.legend()
ax.set_title("Trong vùng train model khớp tốt, ngoài vùng train sai gấp nhiều lần")
plt.show()

Nếu dùng model này để quyết định ngân sách, ta sẽ tin rằng chi 300 triệu mang lại doanh số gấp nhiều lần thực tế. **Khắc phục:** kiểm tra dữ liệu mới có nằm trong vùng train không (chương 05, bài 08); dùng dạng hàm hợp lý về mặt nghiệp vụ (log, bão hoà); thu thập dữ liệu ở vùng cần dự đoán; cảnh báo/từ chối dự đoán khi ngoại suy xa.

## 7. Failure case 6 — Thiếu biến quan trọng (omitted variable bias)

Nếu một biến quan trọng **bị bỏ sót** và nó tương quan với feature đang có, hệ số của feature đó sẽ "gánh" luôn ảnh hưởng của biến bị bỏ → hệ số **sai lệch có hệ thống**.

In [ ]:
m_without = LinearRegression().fit(df[["distance_to_center_km"]], price)
m_with = LinearRegression().fit(X, price)
print(f"Hệ số distance (không có quận)      : {m_without.coef_[0]:.3f} tỷ/km")
print(f"Hệ số distance (có quận + feature khác): {m_with.coef_[list(X.columns).index('distance_to_center_km')]:.3f} tỷ/km")

Khi bỏ quận, khoảng cách "gánh" luôn hiệu ứng nhà ở **trung tâm** (Quận 1, 3 vừa gần vừa đắt) → ảnh hưởng của khoảng cách bị phóng đại nhiều lần. Đây cũng là lý do hệ số hồi quy **không** phải hiệu ứng nhân quả (chương 38).

## 8. Checklist chẩn đoán model hồi quy

- [ ] So sánh với **baseline** (dự đoán bằng trung bình/median).
- [ ] **Residual vs fitted**: có hình cong? hình phễu?
- [ ] **Q-Q plot / histogram residual**: đuôi dày, lệch?
- [ ] **Residual vs từng feature**: feature nào cần biến đổi phi tuyến?
- [ ] **Outlier / leverage / Cook's distance**: điểm nào chi phối model?
- [ ] **VIF**: đa cộng tuyến (quan trọng nếu cần diễn giải)?
- [ ] Dữ liệu dự đoán có nằm trong **vùng train**?
- [ ] Hệ số có **hợp lý** về dấu và độ lớn với kiến thức nghiệp vụ?

## 🧠 Kiểm tra nhanh

1. Residual vs fitted có hình phễu mở rộng. Vấn đề gì và cách khắc phục phổ biến nhất?
2. Vì sao một điểm có leverage cao nguy hiểm hơn một outlier nằm giữa vùng dữ liệu?
3. VIF = 25 nói lên điều gì? Có ảnh hưởng tới độ chính xác dự đoán không?
4. Model được train với chi phí quảng cáo 0–40 triệu. Có nên dùng nó để dự đoán doanh số khi chi 300 triệu không? Vì sao?

## 📚 Tài liệu tham khảo

- ISLR, §3.3.3 *Potential Problems*
- Fox — *Regression Diagnostics* (2nd ed., 2019)
- statsmodels — `OLSInfluence` (leverage, Cook's distance)